# ViFeedback — Cycle 5 H12′ on Kaggle

Trains the four remaining seeds of **H12′** (`configs/experiments/cycle5.yaml` v6): PhoBERT-base with
two heads (the UIT-VSFC head is served, a NEU-ESC head is trained on 18,113 in-scope NEU-ESC train
posts) and H10b's anchored consistency on real-typing pairs. Seed 42 was trained on the laptop and
passed the declared eligibility check (UIT-VSFC +0.0104, NEU-ESC validation +0.0611).

| Part | What | GPU time (T4) | Needed |
|---|---|---|---|
| Setup and checks | clone the pinned commit, data, restorer, fingerprints, smoke run | ~10 min | always |
| **A** | train seeds 1337, 2024, 7, 31337 and write their labels on the confirmation data | ~1.5–2 h | always |
| **B** | the five controls (H10b) and the seed-42 candidate: labels on the confirmation data | ~15 min | only if the checkpoint dataset is attached; otherwise the laptop does it |
| Package | `h12p_results.zip` in the Output tab | 1 min | always |

**Before you run** (owner's guide, in Vietnamese: `docs/HUONG_DAN_THU_CONG.md` § 11):

1. *Settings → Accelerator*: **GPU T4** (one GPU is used). *Settings → Internet*: **on**.
2. *Add-ons → Secrets*: a secret named **`HF_TOKEN`**, from the Hugging Face account that accepted
   NEU-ESC's conditions (the dataset is gated). The token is never printed.
3. Optional, for part B: attach the private dataset holding the six checkpoint folders.
4. *Save Version → Save & Run All (Commit)*, so it runs in the background (up to 12 h).
5. Keep the notebook **private**. Its outputs hold no corpus text, but the session downloads
   UIT-VSFC, NEU-ESC and ViLexNorm under their licences.

**What comes back.** `h12p_results.zip`: summaries, metrics and predicted *labels* (no text), the
run folders and the new registry rows. On the laptop: `vifeedback study h12p-import h12p_results.zip`,
then `study h12p-status` and `study h12p-confirm`.

## 1. Environment

In [ ]:
import os
import pathlib
import shutil
import socket
import subprocess
import sys
import time

import torch

COMMIT = "2f4c4ec3cd20de8afedce28eed0ca7bf9acc2fcd"  # the code H12' runs; pinned
REPO_URL = "https://github.com/ThanhDatVN/ViFeedback-NLP-Service.git"
WORK = pathlib.Path(
    "/tmp/repo"
)  # not /kaggle/working: corpus files and checkpoints stay out of the output
SEEDS_A = [1337, 2024, 7, 31337]

assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU, then run again"
p = torch.cuda.get_device_properties(0)
print(
    f"{p.name}  {p.total_memory / 2**30:.1f} GiB  torch {torch.__version__}  GPUs {torch.cuda.device_count()}"
)
socket.create_connection(("huggingface.co", 443), timeout=10).close()
print("internet: ok")

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN: loaded (not printed)")
except (
    Exception
) as e:  # NEU-ESC is gated: without the token the data step stops with a clear message
    print(f"HF_TOKEN: not found ({type(e).__name__}); add it under Add-ons -> Secrets")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one GPU, as on the laptop


def run(*args, check=True):
    """Run a command in the repository, streaming its output without progress bars."""
    print(">", " ".join(map(str, args)), flush=True)
    t0 = time.time()
    proc = subprocess.Popen(
        [str(a) for a in args],
        cwd=WORK,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    noise = (
        "it/s]",
        "s/it]",
        "UNEXPECTED",
        "MISSING",
        "Notes:",
        "can be ignored",
        "newly initialized",
    )
    for line in proc.stdout:
        if not any(n in line for n in noise):
            print(line, end="", flush=True)
    rc = proc.wait()
    print(f"  [exit {rc} after {time.time() - t0:.0f} s]", flush=True)
    if check and rc:
        raise RuntimeError(f"command failed with exit code {rc}")
    return rc


VF = [sys.executable, "-m", "vifeedback.cli"]

## 2. The pinned code

In [ ]:
if WORK.exists():
    shutil.rmtree(WORK)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(WORK)], check=True)
subprocess.run(["git", "checkout", "--quiet", COMMIT], cwd=WORK, check=True)
head = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=WORK, capture_output=True, text=True
).stdout.strip()
assert head == COMMIT, (head, COMMIT)
print("repository at", head)

# The package without its dependencies, so pip cannot replace Kaggle's CUDA torch; then what
# Kaggle lacks. pyvi is pinned: the segmentation must match the laptop's (checked in section 3).
run(sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", ".")
run(
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "pyvi==0.1.1",
    "typer>=0.12",
    "pyyaml",
    "py-cpuinfo",
    "psutil",
)
for name in [n for n in list(sys.modules) if n == "vifeedback" or n.startswith("vifeedback.")]:
    del sys.modules[name]
sys.path.insert(0, str(WORK / "src"))
os.chdir(WORK)
import importlib.metadata as md  # noqa: E402

print({p: md.version(p) for p in ("torch", "transformers", "numpy", "pandas", "pyvi")})

## 3. Data, and checks that this machine sees the same inputs as the laptop

Each check stops the notebook on a mismatch rather than training on different data:
- UIT-VSFC and the external corpora are pinned by revision and SHA-256 (`configs/data`);
- the pyvi segmentation reproduces the laptop's counts (underscores added, tokens) per split;
- the diacritic restorer rebuilt from UIT-VSFC train is byte-identical to the released one;
- the 3,000 held-out NEU-ESC posts and the ViLexNorm splits equal the committed indices.

In [ ]:
import json

run(*VF, "data", "fetch")
run(*VF, "data", "variants", "--name", "seg_pyvi")
meta = json.loads((WORK / "data/processed/seg_pyvi/meta.json").read_text(encoding="utf-8"))
expected = {"train": (35178, 128382), "validation": (4745, 16898), "test": (9649, 35387)}
got = {
    s: (v["underscores_added"], v["segmented_whitespace_tokens"]) for s, v in meta["splits"].items()
}
assert got == expected, f"pyvi segmentation differs from the laptop: {got} vs {expected}"
print("pyvi segmentation: identical counts")

run(*VF, "data", "fetch-external", "--name", "vilexnorm")
run(*VF, "data", "fetch-external", "--name", "neu_esc")  # gated: needs HF_TOKEN
run(*VF, "study", "h12p-assets")  # restorer rebuilt; refused unless its SHA-256 is the released one
run(*VF, "study", "h12p-split")  # refuses a held-out split that differs from the committed one

from vifeedback.training import h12p as H  # noqa: E402

shapes = {s: H.teacher_q_rows(s).shape for s in ("train", "confirm", "dev")}  # checks the splits
assert shapes == {"train": (6035, 3), "confirm": (1500, 3), "dev": (837, 3)}, shapes
print("teacher targets and ViLexNorm splits: as committed")

## 4. Smoke run (one epoch on 256 examples, writes nothing)

In [ ]:
run(*VF, "study", "h12p-run", "--seed", str(SEEDS_A[0]), "--smoke")

## 5. Part A: the four seeds

About 20–30 minutes each on a T4. A seed whose summary and predictions already exist is skipped, so
re-running the cell after an interruption continues where it stopped (within the same session).

In [ ]:
OUT = WORK / "results/studies/cycle5/h12p"
for seed in SEEDS_A:
    d = OUT / f"two_heads_anchored-s{seed}"
    if (d / "summary.json").exists() and (d / "predictions_confirm.csv").exists():
        print(f"seed {seed}: done already")
        continue
    run(*VF, "study", "h12p-run", "--seed", str(seed))
    torch.cuda.empty_cache()

## 6. Part B (optional): the controls and the seed-42 candidate

Needs the private dataset with the six checkpoint folders, named exactly as on the laptop:
`p15-sent-phobert-base-seg_pyvi-h10b-anchored_orig-s{42,1337,2024,7,31337}-*-ckp` and
`p16-sent-phobert-base-seg_pyvi-h12p-two_heads_anchored-s42-2842e32b-ckp`. Without it, this cell
says so and the laptop writes these predictions later (`study h12p-control`, `study h12p-predict`).

In [ ]:
from vifeedback.constants import SEEDS
from vifeedback.training import consistency as C

wanted = [C.config("anchored_orig", s).run_id("ckpt") for s in SEEDS]
wanted.append(H.config(42).run_id("ckpt"))
found = {}
for cfg_file in pathlib.Path("/kaggle/input").rglob("config.json"):
    if cfg_file.parent.name in wanted:
        found[cfg_file.parent.name] = cfg_file.parent
missing = [w for w in wanted if w not in found]
if missing:
    print("part B skipped; checkpoints not attached:", *missing, sep="\n  ")
else:
    (WORK / "models").mkdir(exist_ok=True)
    for name, src in found.items():
        dst = WORK / "models" / name
        if not dst.exists():
            dst.symlink_to(src, target_is_directory=True)
    for s in SEEDS:
        run(*VF, "study", "h12p-control", "--seed", str(s))
    run(*VF, "study", "h12p-predict", "--seed", "42")
run(*VF, "study", "h12p-status", check=False)

## 7. Package the results (no corpus text)

Only files the runs added or changed under `results/` go into the zip, plus the new registry rows.
Every CSV is checked to hold labels and numbers only.

In [ ]:
import csv
import io
import zipfile

import pandas as pd

status = subprocess.run(
    ["git", "status", "--porcelain", "--untracked-files=all", "results/"],
    cwd=WORK,
    capture_output=True,
    text=True,
    check=True,
).stdout.splitlines()
changed = [line[3:].strip() for line in status]
allowed = ("results/studies/cycle5/h12p/", "results/runs/p16-")
files = [f for f in changed if f.startswith(allowed)]
ignored = [f for f in changed if not f.startswith(allowed) and f != "results/registry.csv"]
if ignored:
    print("not packaged (outside the H12 results):", *ignored, sep="\n  ")


def text_free(path: pathlib.Path) -> bool:
    if path.suffix == ".csv":
        with open(path, encoding="utf-8") as fh:
            rows = csv.reader(fh)
            header = next(rows)
            if any(
                h.lower() in ("text", "raw", "sentence", "original", "normalized") for h in header
            ):
                return False
            return all(len(cell) <= 40 for row in rows for cell in row)
    return path.suffix in (".json", ".yaml", ".png")


bad = [f for f in files if not text_free(WORK / f)]
assert not bad, f"files that may hold text: {bad}"

head_reg = subprocess.run(
    ["git", "show", "HEAD:results/registry.csv"],
    cwd=WORK,
    capture_output=True,
    text=True,
    check=True,
).stdout
old_ids = set(pd.read_csv(io.StringIO(head_reg)).run_id)
reg = pd.read_csv(WORK / "results/registry.csv")
new_rows = reg[~reg.run_id.isin(old_ids)]

out = pathlib.Path("/kaggle/working/h12p_results.zip")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(WORK / f, f)
    z.writestr("registry_rows.csv", new_rows.to_csv(index=False))
print(
    f"{out}  {out.stat().st_size / 1e6:.2f} MB  {len(files)} files, {len(new_rows)} registry rows"
)

## 8. What the seeds scored on the development sets (no held-out label is read here)

In [ ]:
rows = []
for d in sorted(OUT.glob("*-s*")):
    f = d / "summary.json"
    if f.exists():
        s = json.loads(f.read_text(encoding="utf-8"))
        rows.append(
            {
                "model": d.name,
                "uit_validation": s["sets"]["uit_validation"]["macro_f1"],
                "uit_stripped": s["sets"]["uit_validation_stripped"]["macro_f1"],
                "neu_validation": s["sets"]["neu_validation"]["macro_f1"],
                "vilexnorm_dev_agreement": s["vilexnorm_dev"]["agreement"],
                "best_epoch": s.get("best_epoch"),
            }
        )
pd.DataFrame(rows).round(4)